# **Creación de dataset EMG**

## Lectura de los datos EMG

In [1]:
import numpy as np
import pandas as pd

array1 = np.genfromtxt("./señal mano.txt", delimiter=",",skip_header = 2)
array2 = np.genfromtxt("./señal tobillo.txt", delimiter=",",skip_header = 2)
Fs=100


array1[:,1]= array1[:,1] - np.mean(array1[:,1])
array1[:,0] = np.arange(0, len(array1[:,0]))
mreposo = array1[300:1600,1]
mflexleve = array1[1800:2800,1]
mflexbrusca = array1[2800:3800,1]
mcontr_isom = array1[3800:5000,1]

array2[:,1]= array2[:,1] - np.mean(array2[:,1])
array2[:,0] = np.arange(0, len(array2[:,0]))
treposo = array2[0:1700,1]
tmovleve = array2[6400:8100,1]


## Pasando a data tabular estilo Sklearn

In [2]:
def deefe(array:np.ndarray,ntarget:int,col:int=100,Fs:int=Fs) -> tuple[np.ndarray, np.ndarray]:
  Ts=1/Fs
  n = np.arange(0,array.shape[0])  # t = n*Ts
  t = n*Ts

  #Pasamos las observaciones a filas correspondientes a una variable t y d_sensor
  st_sensor = np.concatenate((t.reshape(-1,1),  array.reshape(-1,1)), axis=1)
  #Creamos el data frame con las varibles t y d_sensor
  df = pd.DataFrame(st_sensor, columns=["t","d_sensor"])
  df.head()
  #Establecemos t como index 
  df = df.set_index("t")

  d_obs = df[["d_sensor"]].values.reshape(int(array.shape[0]/col),col)
  target = np.repeat(ntarget, d_obs.shape[0])
  return d_obs,target

obs_mreposo,target_mreposo=deefe(mreposo,0)
obs_mflexleve,target_mflexleve=deefe(mflexleve,1)
obs_mflexbrusca,target_mflexbrusca=deefe(mflexbrusca,2)
obs_mcontr_isom,target_mcontr_isom=deefe(mcontr_isom,3)

obs_treposo,target_treposo=deefe(treposo,4)
obs_tmovleve,target_tmovleve=deefe(tmovleve,5)

## Descripción de categoría de los ejercicios realizado en la clase de ECG

| Descripción | Categoría |
|----------|----------|
|Señal en reposo (mano)                     |0|
|Señal en flexión leve (mano)               |1|
|Señal en flexión brusca (mano)             |2|
|Señal en contracción isométrica (mano)     |3|
|Señal en reposo                            |4|
|Señal en movimiento leve                   |5|

## Creación del diccionario

In [3]:
emg = {"base": np.concatenate([obs_mreposo,obs_mflexleve,obs_mflexbrusca,obs_mcontr_isom,obs_treposo,obs_tmovleve]), "target": np.concatenate([target_mreposo,target_mflexleve,target_mflexbrusca,target_mcontr_isom,target_treposo,target_tmovleve])}
emg

{'base': array([[  4.09882747,  -5.90117253,  10.09882747, ...,  10.09882747,
          -1.90117253,   2.09882747],
        [  6.09882747,  -4.90117253,  13.09882747, ...,   9.09882747,
           1.09882747,  -3.90117253],
        [  3.09882747,  -4.90117253,   7.09882747, ...,   2.09882747,
           2.09882747,  -3.90117253],
        ...,
        [  2.31836518,   3.31836518,   9.31836518, ...,   8.31836518,
          15.31836518,  -2.68163482],
        [  0.31836518,  -8.68163482,   6.31836518, ...,  16.31836518,
           1.31836518,   0.31836518],
        [  1.31836518,   4.31836518, -25.68163482, ...,  -4.68163482,
           1.31836518,  -4.68163482]]),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5,
        5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5])}

## Exportar el diccionario a un archivo .pkl

In [4]:
import pickle
with open('dataset_EMG.pkl', 'wb') as f:
    pickle.dump(emg, f)